<a href="https://colab.research.google.com/github/VedanthParth/AI-Project/blob/pathtrace-rebuild/ml/notebooks/pathtrace_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PathTrace: full experiment grid on Colab

Runs every experiment behind the paper's tables and figures on a GPU, from the `pathtrace-rebuild` branch.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

Everything (data, checkpoints, results) is saved to Google Drive under `MyDrive/pathtrace`, and each step skips work that is already finished. If Colab disconnects, reconnect and run all cells again: the grid picks up where it stopped.

| Dataset | What runs | Why |
| --- | --- | --- |
| `junyi` (random negatives) | simulators, 4 baselines, 3 model variants × 5 seeds, RL × 5 seeds, RL sensitivity, DKT gain | Main results and ablations |
| `junyi-popneg` (popularity-sampled negatives) | baselines, pointer × 3 seeds, RL × 3 seeds, DKT gain | Harder candidate sets |
| `junyi-paper` (2,000 students, last 3 concepts each) | pointer × 3 seeds | Bridge to the paper's original setup |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Drive, code and dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/pathtrace'
REPO = '/content/AI-Project'
BRANCH = 'pathtrace-rebuild'
import os
os.makedirs(ROOT, exist_ok=True)

In [14]:
import os
if os.path.exists(REPO):
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} https://github.com/VedanthParth/AI-Project.git {REPO}
!git -C {REPO} log --oneline -1
!pip install -q -e {REPO}/ml

3fd1449 (HEAD -> pathtrace-rebuild, origin/pathtrace-rebuild) Make matplotlib a required dependency
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pathtrace (pyproject.toml) ... done


## 2. Data (downloaded once into Drive; no Kaggle account needed)

In [15]:
RAW = f'{ROOT}/data/raw/junyi'
if not os.path.exists(f'{RAW}/Log_Problem.csv'):
    os.makedirs(RAW, exist_ok=True)
    !curl -sL -o /content/junyi.zip https://www.kaggle.com/api/v1/datasets/download/junyiacademy/learning-activity-public-dataset-by-junyi-academy
    !unzip -q -o /content/junyi.zip -d {RAW} && rm /content/junyi.zip
!ls -la {RAW}

total 2956559
drwx------ 2 root root       4096 Oct  3 03:19 .
drwx------ 3 root root       4096 Oct  3 03:19 ..
-rw------- 1 root root     389397 May 13  2020 Info_Content.csv
-rw------- 1 root root    6363597 May 13  2020 Info_UserData.csv
-rw------- 1 root root 3020754300 May 13  2020 Log_Problem.csv


In [16]:
PROC = f'{ROOT}/data/processed'
LOG = f'{RAW}/Log_Problem.csv'
datasets = {
    'junyi': '',
    'junyi-popneg': '--negatives popularity',
    'junyi-paper': '--windows last --max-students 2000',
}
for name, flags in datasets.items():
    if not os.path.exists(f'{PROC}/{name}/meta.json'):
        !python -m pathtrace prepare --log {LOG} --out {PROC}/{name} {flags}
    else:
        print(f'{name}: already prepared')

junyi: already prepared
junyi-popneg: already prepared
junyi-paper: already prepared


## 3. Experiment grids

`BATCH = 256` keeps the GPU busy; every final number in the paper should come from these runs, so report this batch size in the paper.

The main grid is the long step: each pointer-network run trains for up to 50 epochs, and there are 3 variants per seed. Watch the per-epoch time it prints in the first run to estimate the total. If Colab time is short, set `SEEDS = '0 1 2'` (3 seeds instead of 5) and skip the two optional grids. After a disconnect, run all cells again: finished runs are skipped.

In [17]:
RUNS = f'{ROOT}/runs'
CONTENT = f'{RAW}/Info_Content.csv'
BATCH = 256
SEEDS = '0 1 2'
COMMON = f'--content {CONTENT} --batch-size {BATCH} --device cuda'

In [ ]:
# Main results: 3 variants x SEEDS, RL per seed, sensitivity runs, DKT learning gain.
!python -m pathtrace grid --data {PROC}/junyi --runs {RUNS}/junyi {COMMON} --seeds {SEEDS}

[skip] bkt: already done
[skip] prereq: already done
[skip] dkt: already done
[skip] baseline random: already done
[skip] baseline popularity: already done
[skip] baseline markov: already done
[skip] baseline gru_next_item: already done
[skip] pointer-seed0: already done
[skip] pointer-nokt-seed0: already done
[skip] pointer-noattn-seed0: already done
[skip] pointer-seed1: already done
[skip] pointer-nokt-seed1: already done
[run]  pointer-noattn-seed1
126,273 parameters, 340,313 training examples, device cuda
epoch   1  train path 2.1237 kt 0.5359  |  val path 1.5578  ndcg 0.6741  hr 0.9049  kt-auc 0.7356  (115.9s)
epoch   2  train path 1.4004 kt 0.5260  |  val path 1.2893  ndcg 0.7523  hr 0.9338  kt-auc 0.7397  (114.7s)
epoch   3  train path 1.2242 kt 0.5243  |  val path 1.1982  ndcg 0.7771  hr 0.9432  kt-auc 0.7417  (113.5s)
epoch   4  train path 1.1346 kt 0.5232  |  val path 1.1433  ndcg 0.7930  hr 0.9481  kt-auc 0.7431  (112.1s)
epoch   5  train path 1.0770 kt 0.5224  |  val path 

In [ ]:
# Remaining supervised runs for 3 seeds, one background job each (no RL/gain; rerun the main cell after they finish).
!nproc
G = f'python -m pathtrace grid --data {PROC}/junyi --runs {RUNS}/junyi {COMMON} --no-rl --no-gain'
!nohup {G} --seeds 1 --variants pointer-noattn > {ROOT}/log_1.txt 2>&1 &
!nohup {G} --seeds 2 --variants pointer > {ROOT}/log_2.txt 2>&1 &
!nohup {G} --seeds 2 --variants pointer-nokt > {ROOT}/log_3.txt 2>&1 &
!nohup {G} --seeds 2 --variants pointer-noattn > {ROOT}/log_4.txt 2>&1 &

In [ ]:
# Progress of the background jobs; rerun any time. Python buffers the log files, so read each run's history.json instead.
import json, glob
for d in sorted(glob.glob(f'{RUNS}/junyi/pointer*-seed[12]')): h = json.load(open(f'{d}/history.json')) if os.path.exists(f'{d}/history.json') else []; print(os.path.basename(d).ljust(22), 'DONE' if os.path.exists(f'{d}/metrics.json') else (f"epoch {h[-1]['epoch']}  ndcg {h[-1]['val_ndcg']:.4f}  ({h[-1]['seconds']}s/epoch)" if h else 'starting'))
!echo; echo "grid processes running: $(pgrep -fc 'pathtrace grid')"; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

In [ ]:
# Optional. Harder candidate sets: pointer network and RL, 3 seeds.
!python -m pathtrace grid --data {PROC}/junyi-popneg --runs {RUNS}/junyi-popneg {COMMON} --variants pointer --seeds 0 1 2 --no-sensitivity

In [ ]:
# Optional. The paper's original setup, for comparison with its reported numbers.
!python -m pathtrace grid --data {PROC}/junyi-paper --runs {RUNS}/junyi-paper {COMMON} --variants pointer --seeds 0 1 2 --no-rl --no-gain

## 4. Tables and figures for the paper

In [ ]:
for name in datasets:
    if os.path.exists(f'{RUNS}/{name}'):
        !python -m pathtrace report --data {PROC}/{name} --runs {RUNS}/{name} --out {ROOT}/results/{name}
!ls -R {ROOT}/results

The `results/` folder in Drive holds the tables (Markdown, LaTeX, CSV) and figures (PNG). The next cell shows the main ones here, so you can screenshot or download them.

In [ ]:
from IPython.display import Image, Markdown, display
out = f'{ROOT}/results/junyi'
display(Markdown(open(f'{out}/main_table.md').read()))
for fig in ['fig_methods.png', 'fig_training.png', 'fig_rl.png']:
    print(fig)
    display(Image(f'{out}/{fig}'))

## 5. App bundle from the full-data runs

Packs the seed-0 pointer network, its simulators and 300 held-out test students for the demo app. Download `bundle.zip` from `MyDrive/pathtrace`, unzip it, and replace the repo's `backend/bundle` folder with it.

In [ ]:
!python -m pathtrace export --data {PROC}/junyi --runs {RUNS}/junyi --checkpoint pointer-seed0 --content {CONTENT} --out /content/bundle
!cd /content && zip -qr bundle.zip bundle && cp bundle.zip {ROOT}/bundle.zip && ls -la {ROOT}/bundle.zip